# Producer Pattern 2 of 5 — Callback-Confirmed Publishing
### A junior-engineer-friendly, production-aware Kafka notebook

**One-line idea:** *Send asynchronously, but learn the final delivery outcome of every accepted message.*

This follows **Pattern 1 — Fire-and-Forget**. The fundamental difference is **one new argument**: `on_delivery=...`. We will build the pattern gradually rather than jumping straight into a large class.

**Who should use this?** Junior Python developers learning Kafka; engineers building order, inventory, and event-driven systems.

**You will learn to:**

1. Explain what **queued**, **acknowledged**, **failed**, and **consumed** mean.
2. Write a delivery callback and know *when it actually executes*.
3. Send many messages asynchronously without `flush()` after each one.
4. Correlate each callback with its business event.
5. Handle broker failures and `BufferError` without infinite retries.
6. Understand `acks`, idempotence, and the limits of delivery confirmation.
7. Build, run, and test a small **production-oriented reusable producer**.

**Prerequisites:** Python basics (functions, dictionaries, `try`/`except`) and a Kafka broker exposed at `localhost:9092`. Your existing `kafka-lab` Docker environment works.

> **Lab safety:** Topic names in this notebook are dedicated to this learning exercise. We **never delete/recreate** a topic, and we do not intentionally interrupt a healthy broker.

## Learning map — our e-commerce example

An API receives an order and emits an `order.created` event to Kafka. A downstream service may read it later.

```text
    Order API (our Python process)
         |
         | producer.produce(..., on_delivery=callback)
         v
    Producer local queue
         |
         | librdkafka background networking
         v
    Kafka broker — accepts message according to acks policy
         |
         | Broker response processed by background networking
         v
    Delivery result becomes available
         |
         | producer.poll() OR producer.flush()
         v
    Your on_delivery(error, message) callback
         |
         +--> success: log/metric/mark delivery outcome
         |
         +--> failure: log/metric/escalate or recover

    Kafka Consumer (later, separate)
         |
         +--> consumer.poll() reads topic records
```

**Never mix these statements up:**

| Statement | True? |
|---|---|
| `produce()` returned → Kafka stored it | **No** — only locally accepted |
| `on_delivery(error=None)` ran → broker acknowledged | **Yes**, according to producer's `acks` policy |
| Broker acknowledged → consumer processed it | **No** — consumption is separate |
| `poll()` sends the message | **No** — producer networking does |
| `flush()` returns `0` → every message succeeded | **No** — terminal failures also finish outstanding work |

---
# Section 0 — Prepare the lab

**Objective:** Connect to an existing Kafka broker and create a dedicated topic *without deleting existing data*.

**Environment:** This notebook targets the `confluent-kafka` Python package (the `librdkafka`-based client).

If needed, install it in your notebook environment and restart the kernel:

```python
%pip install -U confluent-kafka
```

You can keep using your local Docker broker from the previous notebook. If it is not running, start it using your existing local setup **before running the next cell**.

In [1]:
import json
import logging
import os
import time
import uuid
from collections import deque
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import confluent_kafka
from confluent_kafka import KafkaException, Producer
from confluent_kafka.admin import AdminClient, NewTopic

# Change through an environment variable if Kafka is not on localhost.
BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
TOPIC = "callback_confirmed_orders_lab"

print("confluent-kafka:", confluent_kafka.__version__)
print("Bootstrap broker:", BOOTSTRAP)

confluent-kafka: 2.16.0
Bootstrap broker: localhost:9092


### 0.1 Verify the broker and ensure the topic exists

Why a helper? We want the rest of the notebook to focus on **delivery callbacks**, not repeat Admin API setup.

This helper uses `replication_factor=1` because your local lab has one broker. Production topics generally need multiple brokers/replicas.

**Unlike `recreate_topic()`, this function will not destroy a topic or its messages.**

In [2]:
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})


def ensure_topic(name: str, partitions: int = 3) -> None:
    """Create a lab topic only if it does not already exist."""
    metadata = admin.list_topics(timeout=10)  # Also verifies broker connectivity.

    if name in metadata.topics:
        print(f"Topic already exists: {name}")
        return

    request = NewTopic(name, num_partitions=partitions, replication_factor=1)
    future = admin.create_topics([request])[name]
    future.result(timeout=15)
    print(f"Created topic: {name} ({partitions} partitions)")


try:
    ensure_topic(TOPIC)
except Exception as exc:
    print("Could not reach/create topic on Kafka:", exc)
    print("Check that your local broker is running and accessible at", BOOTSTRAP)
    raise

Created topic: callback_confirmed_orders_lab (3 partitions)


---
# Section 1 — Your very first callback

**Objective:** Add a callback to one `produce()` call. This is the *smallest possible* callback-confirmed producer.

### Our courier analogy

1. **`produce()`** — Hand a parcel to the courier desk.
2. **Background networking** — Courier transports it without you waiting.
3. **Kafka acknowledgement** — Destination accepts it according to the agreed rules.
4. **Delivery report** — Courier system records success or final failure.
5. **`producer.poll()` / `flush()`** — Your application processes the report; your callback runs.

The callback has **two parameters**:

- `error`: `None` for successful delivery; a Kafka error object for final failure.
- `message`: metadata about the message (e.g. topic, partition, offset) and its key/value.

It is a **final delivery callback**, not a callback for every internal retry.

In [3]:
# STEP 1 — A function that receives Kafka's final delivery result.
def delivery_report(error, message):
    """Called by poll()/flush() when this message succeeds or finally fails."""
    if error is not None:
        print("❌ Delivery failed:", error)
    else:
        print(
            "✅ Broker acknowledged:",
            f"topic={message.topic()}",
            f"partition={message.partition()}",
            f"offset={message.offset()}",
        )


# STEP 2 — Create the Kafka producer. Reuse an instance for many messages.
producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
})

# STEP 3 — Queue the message. This does NOT wait for the broker response.
producer.produce(
    topic=TOPIC,
    key="order-101",
    value=b'{"order_id":101,"status":"CREATED"}',
    on_delivery=delivery_report,          # <-- New compared with fire-and-forget!
)
print("The message was accepted into the producer's local queue.")

# STEP 4 — Wait for outstanding deliveries; this ALSO executes callbacks.
remaining = producer.flush(timeout=15)
print("Still outstanding after flush:", remaining)

The message was accepted into the producer's local queue.
✅ Broker acknowledged: topic=callback_confirmed_orders_lab partition=1 offset=0
Still outstanding after flush: 0


**What to notice:** The application prints **“accepted into the local queue”** before it receives the final delivery report. A callback with `error is None` tells us Kafka acknowledged the record under our `acks` setting.

**What it does *not* tell us:** Whether a consumer subsequently read it, validated it, updated a database, or completed the business workflow.

**Checkpoint:** Why doesn't `produce()` print “delivered” immediately? Because actual transmission and broker acknowledgements happen **asynchronously**.

---
# Section 2 — Exactly when does the callback run?

**Objective:** Show that networking continues without `poll()`, while your Python delivery callback normally waits until events are serviced.

We will intentionally *not* call `poll()` for a brief period. **This is a learning experiment, not the recommended steady-state production loop.**

**Possible output:** Callback count starts at zero, stays zero during sleep, then increases after `poll(0)` and/or `flush()`. A single non-blocking `poll(0)` may process **some or all** ready callbacks, or none if results aren't ready.

In [ ]:
callback_results = []


def track_callback(error, message):
    callback_results.append("failed" if error is not None else "delivered")


experiment = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all"})

# A — Submit 10 messages. No poll() and no flush() yet.
for number in range(10):
    experiment.produce(TOPIC, value=f"experiment-{number}", on_delivery=track_callback)

print("A — Immediately after produce():", len(callback_results), "callbacks")

# B — Producer networking can work while our Python thread sleeps.
time.sleep(1)
print("B — After sleeping without poll():", len(callback_results), "callbacks")

# C — Process available delivery events without waiting.
processed_events = experiment.poll(0)
print("C — poll(0) serviced events:", processed_events)
print("C — Callbacks executed so far:", len(callback_results))

# D — Finish any remaining outstanding work.
remaining = experiment.flush(timeout=15)
print("D — Total callbacks after flush():", len(callback_results))
print("D — Delivery results:", callback_results)
print("D — Outstanding:", remaining)

### Section 2 recap

- `produce()` registers a callback and submits a message **locally**.
- Background networking sends the message and receives Kafka's response.
- `poll()` **serves** delivery events; it does not send messages itself.
- `flush()` serves delivery events **while waiting** for outstanding work.
- `len(producer)` is not a reliable “number of messages the broker hasn't stored” metric. It may include queued events and requests.

**Engineer takeaway:** In long-running processes, **service producer events regularly**. Use `flush()` primarily at orderly shutdown or deliberate batch boundaries, not after every message.

---
# Section 3 — Send a batch asynchronously (the normal producer loop)

**Objective:** Publish many order events without blocking for each acknowledgement, while *counting real callback outcomes*.

This is the pattern most engineers should remember:

```text
for each event:
    produce(event, on_delivery=callback)  # queue asynchronously
    poll(0)                               # service ready events

flush(timeout)                            # drain before shutdown
```

We record **queued**, **delivered**, and **failed** separately because they mean different things.

In [ ]:
stats = {"queued": 0, "delivered": 0, "failed": 0}


def count_delivery(error, message):
    if error is None:
        stats["delivered"] += 1
    else:
        stats["failed"] += 1
        print("Delivery failure:", error)


def make_simple_order(order_id: int) -> dict:
    """A realistic event payload we can serialize to JSON."""
    return {
        "event_type": "order.created",
        "order_id": order_id,
        "customer_id": 2001,
        "total_cents": 4999,  # Integer money unit; avoid floating-point rounding.
    }


batch_producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
})

for order_id in range(1, 21):
    order = make_simple_order(order_id)

    # JSON -> bytes before sending.
    payload = json.dumps(order).encode("utf-8")

    batch_producer.produce(
        TOPIC,
        key=str(order_id),    # Same order_id -> same partition in this topic.
        value=payload,
        on_delivery=count_delivery,
    )
    stats["queued"] += 1  # Accepted locally, not yet confirmed.

    batch_producer.poll(0)     # Non-blocking event servicing.

remaining = batch_producer.flush(timeout=15)

print("Queued:", stats["queued"])
print("Delivered:", stats["delivered"])
print("Failed:", stats["failed"])
print("Callbacks still pending:", stats["queued"] - stats["delivered"] - stats["failed"])
print("Producer outstanding after flush:", remaining)

### How to interpret the counters

| Counter | What it tells you |
|---|---|
| `queued` | `produce()` returned successfully (local acceptance) |
| `delivered` | Callback reported Kafka acknowledgement |
| `failed` | Callback reported final delivery failure |
| `queued - delivered - failed` | Your application has not yet processed a final callback for those accepted messages |

**Crucial:** `flush()` returning `0` is **not equivalent** to `failed == 0`. A failure callback counts as completed work, too.

In normal production code, if you need to know whether publishing succeeded, inspect delivery failures rather than inferring success from `flush()` alone.

---
# Section 4 — Which business event did Kafka acknowledge?

**Objective:** Correlate a delivery report to the **specific order event** that triggered it.

Real producers have many requests in flight. Kafka may invoke callbacks later, and callback completion order isn't necessarily the same as the order in which you submitted all the events.

**Professional habit:** Include a unique `event_id` in each payload. Use a stable business key such as `order_id` to control Kafka partitioning. A small callback closure can remember `event_id` without storing every event in a global dictionary.

- `event_id`: identifies **this event**, useful for tracing/deduplication.
- `order_id` Kafka key: groups events for the same order onto the same partition (while the partition count and partitioner remain stable).
- `partition` and `offset`: Kafka's position for a successfully acknowledged record.

In [ ]:
correlated_results = []  # Lab-only: bounded batch; don't keep every result forever in prod.


def make_order_event(order_id: int) -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "event_type": "order.created",
        "schema_version": 1,
        "order_id": order_id,
        "total_cents": 12999,
        "occurred_at": datetime.now(timezone.utc).isoformat(),
    }


def callback_for(event_id: str):
    """Make a callback that remembers which event we are publishing."""
    def on_delivery(error, message):
        result = {
            "event_id": event_id,
            "status": "failed" if error else "delivered",
            "error": str(error) if error else None,
            "partition": message.partition() if not error else None,
            "offset": message.offset() if not error else None,
        }
        correlated_results.append(result)
    return on_delivery


correlation_producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all"})

for order_id in (501, 502, 503):
    event = make_order_event(order_id)

    correlation_producer.produce(
        TOPIC,
        key=str(event["order_id"]),
        value=json.dumps(event).encode("utf-8"),
        on_delivery=callback_for(event["event_id"]),
    )
    correlation_producer.poll(0)

correlation_producer.flush(15)
for result in correlated_results:
    print(result)

**Why closures?** The producer callback receives `(error, message)`; it does **not** receive your Python order dictionary as an additional argument. The nested function remembers the event ID it was created with.

**Important API detail:** The delivery callback's returned `message` may not expose the original message **headers**. Do not rely on `message.headers()` for callback correlation; capture the ID in a closure or decode an appropriate key/value.

**Memory concern:** Our `correlated_results` list is for a tiny notebook experiment. A real producer should emit metrics/logs or write a bounded record of failures, not accumulate millions of callback results in memory.

---
# Section 5 — What a *real delivery failure* looks like

**Objective:** Prove that callbacks report failures, not only successful acknowledgements.

We deliberately connect **a separate producer** to a port where no broker is expected. The live Kafka cluster is untouched.

1. `produce()` can still accept the message **locally**.
2. Background networking cannot reach a broker.
3. The delivery deadline expires.
4. `poll()` / `flush()` serves a **failure callback**.

The example uses a short delivery timeout to keep the lab bounded.

In [ ]:
failure_results = []


def record_failure(error, message):
    if error is None:
        failure_results.append("delivered")
        print("Unexpected successful delivery")
    else:
        failure_results.append("failed")
        print("EXPECTED failure callback:", error)


# We expect this port NOT to contain a Kafka broker.
broken = Producer({
    "bootstrap.servers": "127.0.0.1:19092",
    "delivery.timeout.ms": 4000,
})

broken.produce(
    topic=TOPIC,
    value=b"unreachable-broker-example",
    on_delivery=record_failure,
)
print("produce() accepted the message LOCALLY")

# Wait enough time to allow the message's delivery deadline to expire.
remaining = broken.flush(timeout=8)
print("Callbacks:", failure_results)
print("Remaining:", remaining)

### Two kinds of failure — do not mix them

| Failure | Where detected | Safe next action |
|---|---|---|
| `produce()` raises `BufferError` | Immediately: local queue rejected the message | Wait/service events and retry submitting the **same not-yet-accepted message**, within a deadline |
| `on_delivery(error, message)` reports failure | Later: client reached a final delivery outcome | Record the failure; apply a **business recovery strategy** |

Why not blindly call `produce()` again inside a failed-delivery callback? Some failures are **ambiguous**: the broker may have stored the record even if the producer could not confirm it. An application-level resend can duplicate the business event. Prefer stable event IDs, deduplication, and/or a durable outbox where appropriate.

**What the callback means:** “This message reached a final delivery result from the producer's perspective,” not “the business operation was completed end-to-end.”

---
# Section 6 — Backpressure: handling `BufferError` properly

**Objective:** Handle local producer queue pressure without losing the event silently or retrying forever.

`BufferError` means **the producer's local queue refused another record**. It can happen when the application publishes faster than the producer/network/broker can handle, or when queue limits are small.

`poll()` services events and gives the producer time to progress; it **does not guarantee** space is freed immediately. Do not rely on a fixed one-time retry.

**Recommended pattern:** Retry the `produce()` operation with a short `poll()` and an overall **deadline**. If the deadline is exhausted, surface an error so the caller can decide what to do.

In [ ]:
def queue_with_deadline(client, topic, *, key, value, callback, max_wait_seconds=5):
    """Queue ONE record, retrying only local BufferError up to a deadline.

    Success means accepted into the local producer queue, NOT delivered.
    """
    deadline = time.monotonic() + max_wait_seconds

    while True:
        try:
            client.produce(
                topic=topic,
                key=key,
                value=value,
                on_delivery=callback,
            )
            return  # Local acceptance succeeded.

        except BufferError:
            remaining = deadline - time.monotonic()

            if remaining <= 0:
                raise TimeoutError(
                    "Kafka producer queue stayed full; caller must handle the unsent event"
                )

            # Process events, wait briefly, and let networking keep working.
            client.poll(min(0.25, remaining))

### Deterministic backpressure test (no Kafka broker needed)

A real queue may fill or drain too quickly to reproduce reliably on a healthy laptop. Rather than claim a 2,000-message loop **always** raises `BufferError`, we'll use a tiny fake to show the retry logic deterministically.

This is **testing the application's queue-full handling**, not testing Kafka itself.

In [ ]:
class FakeQueueFullClient:
    """Act like produce() refuses the first N attempts, then accepts."""

    def __init__(self, failures_before_accepting=2):
        self.failures_left = failures_before_accepting
        self.calls = 0

    def produce(self, **kwargs):
        self.calls += 1
        if self.failures_left > 0:
            self.failures_left -= 1
            raise BufferError("Simulated local producer queue full")
        self.last_accepted = kwargs

    def poll(self, seconds):
        print(f"  queue full -> service events for up to {seconds:.2f}s")
        return 0


fake = FakeQueueFullClient(failures_before_accepting=2)
queue_with_deadline(
    fake,
    TOPIC,
    key=b"order-900",
    value=b'{"order_id":900}',
    callback=lambda error, msg: None,
    max_wait_seconds=3,
)

print("Message finally accepted on attempt:", fake.calls)
assert fake.calls == 3

### Production distinction: backpressure vs delivery retry

- **Retry after `BufferError`:** The message was not accepted locally; retrying that enqueue attempt is appropriate, with a deadline.
- **Retry after a final delivery callback error:** A separate decision involving potential duplicates, business idempotency, and durable recovery.
- **Never** sleep/retry forever inside the callback. Keep delivery callbacks fast so they don't block the event-service loop.

**Junior engineer memory:** `produce()` can raise `BufferError`; catch it, service events, retry with a deadline, then escalate.

---
# Section 7 — Reliability configurations worth understanding

**Objective:** Learn the **few settings engineers commonly review** for callback-confirmed publishing, without memorizing the whole config reference.

| Config | Why it matters |
|---|---|
| `bootstrap.servers` | Where to find the Kafka cluster |
| `acks="all"` | Request strongest broker acknowledgement supported by the topic/cluster |
| `enable.idempotence=True` | Avoid duplicates caused by eligible producer retries within the producer's guarantees |
| `delivery.timeout.ms` | Bound how long a message can remain in the producer's delivery process before final failure |
| `linger.ms` | Allow brief batching for throughput, at some latency cost |
| `compression.type` | Reduce network bandwidth; evaluate based on your payload and cluster |

```python
RELIABLE_PRODUCER_CONFIG = {
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
    "delivery.timeout.ms": 30_000,
    "linger.ms": 5,
    # Consider "compression.type": "snappy" or "zstd" after measuring.
}
```

**Critical distinctions:**

- `acks="all"` on a **single-broker, replication-factor-1 lab** does not create replicas or the same fault tolerance as a properly replicated production cluster.
- `enable.idempotence=True` helps protect against duplicates from **automatic producer retries**. It does **not** give end-to-end exactly-once processing or make manual application resends automatically safe.
- A **successful callback** does not prove that a consumer processed the event or that another database committed.
- These values are **learning defaults**, not universally correct SLOs. Production timeouts/batching depend on workload.

---
# Section 8 — Build a reusable, professional producer

**Objective:** Combine everything into a small service-facing abstraction.

Our application should be able to call:

```python
publisher.publish(event)  # enqueue + correlation + bounded queue-full handling
publisher.poll()          # process callbacks during a long-running workload
summary = publisher.finish()  # bounded drain + delivery summary
```

### What good design means here

- **One producer instance** for many events, not one producer per order.
- **Explicit input contract** — require event IDs, event type, and order ID.
- **Serialize at the boundary** — application dictionaries become JSON bytes.
- **Correlation** — every callback knows its `event_id`.
- **Bounded queue retry** — backpressure is handled, but never infinite.
- **Lightweight callbacks** — update counters and log failures without blocking network/event progress.
- **Bounded memory** — store only a small sample of failed events, not every success.
- **Explicit shutdown report** — never mistake `flush()==0` for zero failures.

**Scope:** This implementation is designed for **one publishing/service-event thread**. For multiple application threads, synchronize shared metrics and coordinate which thread services callbacks; `librdkafka` supports thread-safe producer use, but *our Python counters* would need careful handling.

In [ ]:
# This cell defines the reusable production-oriented implementation.
# Read it from top to bottom; the usage cell comes next.

@dataclass(frozen=True)
class DeliverySummary:
    """Snapshot of outcomes for records accepted by this publisher."""
    queued: int
    delivered: int
    failed: int
    awaiting_callback: int
    producer_remaining: int

    @property
    def all_confirmed(self) -> bool:
        """True only when no accepted message failed or remains unresolved."""
        return self.failed == 0 and self.awaiting_callback == 0 and self.producer_remaining == 0


class CallbackConfirmedOrderProducer:
    """Asynchronous Kafka publishing with delivery result tracking.

    Lifecycle:
        create -> publish many -> poll regularly -> finish once

    This tracks *producer delivery*, NOT downstream consumer processing.
    """

    def __init__(
        self,
        bootstrap_servers: str,
        topic: str,
        *,
        max_queue_wait: float = 5.0,
        producer_client=None,
    ):
        self.topic = topic
        self.max_queue_wait = max_queue_wait
        self.log = logging.getLogger("callback_confirmed_producer")
        self._finished = False

        # Allow dependency injection of a fake producer for unit tests.
        # The normal application path creates an actual Kafka client.
        self._client = producer_client if producer_client is not None else Producer({
            "bootstrap.servers": bootstrap_servers,
            "acks": "all",
            "enable.idempotence": True,
            "delivery.timeout.ms": 30_000,
            "linger.ms": 5,
        })

        # Lifetime counters: only ONE app thread services callbacks here.
        self.queued = 0
        self.delivered = 0
        self.failed = 0

        # Keep at most 20 examples for debugging; no unbounded list.
        self.recent_failures = deque(maxlen=20)

    def _delivery_callback(self, event_id: str):
        """Create a callback that knows which business event it belongs to."""

        def on_delivery(error, message):
            # Keep this fast: no blocking I/O, DB writes, or long retry loop.
            if error is None:
                self.delivered += 1
                self.log.debug(
                    "Kafka ack | event_id=%s topic=%s partition=%s offset=%s",
                    event_id,
                    message.topic(),
                    message.partition(),
                    message.offset(),
                )
            else:
                self.failed += 1
                self.recent_failures.append((event_id, str(error)))
                self.log.error(
                    "Kafka delivery failed | event_id=%s error=%s",
                    event_id,
                    error,
                )

        return on_delivery

    def publish(self, event: dict) -> str:
        """Queue one event and return its ID — NOT a delivery receipt.

        A BufferError is retried only until max_queue_wait elapses.
        Other synchronous errors are intentionally propagated to callers.
        """
        if self._finished:
            raise RuntimeError("Cannot publish after finish()")

        required = ("event_id", "event_type", "order_id")
        missing = [field for field in required if field not in event]
        if missing:
            raise ValueError(f"Missing required event fields: {missing}")

        event_id = str(event["event_id"])
        message_key = str(event["order_id"]).encode("utf-8")
        message_value = json.dumps(event, separators=(",", ":")).encode("utf-8")
        callback = self._delivery_callback(event_id)

        # Retrying a LOCAL BufferError is safe: produce() did not accept it.
        deadline = time.monotonic() + self.max_queue_wait

        while True:
            try:
                self._client.produce(
                    topic=self.topic,
                    key=message_key,
                    value=message_value,
                    on_delivery=callback,
                )
                self.queued += 1
                return event_id  # LOCAL acceptance only.

            except BufferError:
                seconds_left = deadline - time.monotonic()
                if seconds_left <= 0:
                    raise TimeoutError(
                        f"Kafka local queue remained full for event_id={event_id}"
                    )
                self._client.poll(min(0.25, seconds_left))

    def poll(self, timeout: float = 0.0) -> int:
        """Process ready producer events; 0 means do not wait."""
        if self._finished:
            raise RuntimeError("Cannot poll after finish()")
        return self._client.poll(timeout)

    def finish(self, timeout: float = 35.0) -> DeliverySummary:
        """Wait for outstanding messages and return truthful final counts.

        Call once as part of orderly shutdown. A nonzero remaining count
        means some delivery outcomes are still unresolved at this point.
        """
        if self._finished:
            raise RuntimeError("finish() has already been called")
        self._finished = True

        remaining = self._client.flush(timeout)
        pending_callbacks = self.queued - self.delivered - self.failed

        return DeliverySummary(
            queued=self.queued,
            delivered=self.delivered,
            failed=self.failed,
            awaiting_callback=pending_callbacks,
            producer_remaining=remaining,
        )

### Read the design from the *outside in*

The large cell becomes manageable if you map each method to one responsibility:

| Component | Responsibility |
|---|---|
| `__init__` | Own and configure a producer; initialize bounded metrics |
| `publish(event)` | Validate, JSON encode, enqueue, handle local queue pressure |
| `_delivery_callback(event_id)` | Count success/failure and correlate with business event |
| `poll()` | Service callbacks during normal publishing |
| `finish()` | Gracefully drain and return an honest report |
| `DeliverySummary.all_confirmed` | Make success criteria explicit instead of checking only `flush()` |

**Why not have `publish()` wait for confirmation?** That would make each send synchronous, reduce throughput, and defeat the benefit of the asynchronous producer pattern. The caller learns results through callbacks and aggregated reporting.

---
# Section 9 — Use the reusable producer in an application

**Objective:** Demonstrate the entire real-world lifecycle, including cleanup in a `finally` block.

This cell deliberately keeps the *business application code* very short. The producer class owns infrastructure concerns; the caller owns business decisions.

We print all outcomes at the end because this is a small notebook. In a service, wire these metrics to your observability system.

In [ ]:
# Configure readable application logs.
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

publisher = CallbackConfirmedOrderProducer(
    bootstrap_servers=BOOTSTRAP,
    topic=TOPIC,
    max_queue_wait=5.0,
)

summary = None
try:
    for order_id in range(1001, 1021):
        event = make_order_event(order_id)

        event_id = publisher.publish(event)
        print("Queued event:", event_id[:8])

        # Normal running pattern: service events periodically.
        publisher.poll(0)

finally:
    # Always attempt a bounded drain, even if publishing raised an exception.
    summary = publisher.finish(timeout=35)

print("\nFinal delivery summary:")
print(asdict(summary))
print("All confirmed:", summary.all_confirmed)
print("Recent failures (up to 20):", list(publisher.recent_failures))

# A real application should report or escalate if any delivery failed,
# or if messages remain unresolved. It should NOT claim business success.
if not summary.all_confirmed:
    print("ACTION REQUIRED: investigate failures or unresolved delivery outcomes")

### Read the output like an engineer

A healthy local broker typically gives:

```text
queued = 20
successfully delivered = 20
failed = 0
awaiting_callback = 0
producer_remaining = 0
all_confirmed = True
```

These are **example values**, not guaranteed output. The topic may have old messages, but only the 20 events sent by this `publisher` instance are counted.

**Important:** Even `all_confirmed=True` is a *producer-delivery* statement, **not proof of exactly-once business processing**.

### In a production service

- Use a durable source of truth or **transactional outbox** for business-critical events that must survive application crashes.
- Alert on delivery failures, unresolved shutdown deliveries, and queue pressure.
- Make downstream consumers idempotent using stable `event_id` values.
- Keep callbacks fast; hand off expensive recovery operations to another component.
- Adapt concurrency and shutdown coordination to your server's threading/async model.

---
# Section 10 — Test producer logic without Kafka

**Objective:** Unit-test your own callback/retry/accounting logic *deterministically*, even when a broker is unavailable.

You should test two different things:

1. **Unit tests:** Your code reacts correctly when the Kafka client reports success, failure, or `BufferError`.
2. **Integration tests:** A real Kafka broker accepts records and callbacks run after `poll()` / `flush()`.

We already performed integration exercises in previous sections. Now we inject a fake producer into the reusable class.

**Note:** Our fake is not Kafka; it is a controlled stand-in for testing application logic.

In [ ]:
class FakeMessage:
    """Only the Message methods our delivery callback actually uses."""

    def __init__(self, topic, partition=0, offset=7):
        self._topic = topic
        self._partition = partition
        self._offset = offset

    def topic(self): return self._topic
    def partition(self): return self._partition
    def offset(self): return self._offset


class FakeProducer:
    """Controllable fake: supports accepted sends, failures and queue pressure."""

    def __init__(self, *, delivery_error=None, full_first=0):
        self.delivery_error = delivery_error
        self.full_first = full_first
        self.queue_attempts = 0
        self.pending = []

    def produce(self, *, topic, key, value, on_delivery):
        self.queue_attempts += 1
        if self.full_first > 0:
            self.full_first -= 1
            raise BufferError("Simulated queue full")
        self.pending.append((topic, on_delivery))

    def poll(self, timeout=0):
        # Execute at most one pending callback for deterministic observation.
        if not self.pending:
            return 0
        topic, callback = self.pending.pop(0)
        callback(self.delivery_error, FakeMessage(topic))
        return 1

    def flush(self, timeout=35):
        while self.pending:
            self.poll(0)
        return 0

In [ ]:
# TEST 1 — Successful asynchronous delivery.
fake_success = FakeProducer()
service = CallbackConfirmedOrderProducer("unused", "fake-topic", producer_client=fake_success)
service.publish(make_order_event(2001))

assert service.queued == 1
assert service.delivered == 0  # Callback hasn't run yet.

service.poll(0)               # Fake now executes the successful callback.
summary = service.finish()
assert summary.all_confirmed
assert summary.delivered == 1
print("PASS: successful callback counted correctly")


# TEST 2 — Delivery failure is never counted as success.
fake_failure = FakeProducer(delivery_error="Simulated delivery failure")
service = CallbackConfirmedOrderProducer("unused", "fake-topic", producer_client=fake_failure)
service.publish(make_order_event(2002))
summary = service.finish()    # Fake executes the failed callback.
assert summary.failed == 1
assert summary.delivered == 0
assert not summary.all_confirmed
print("PASS: failed callback is visible")


# TEST 3 — Local queue full: bounded retry succeeds after pressure clears.
fake_pressure = FakeProducer(full_first=2)
service = CallbackConfirmedOrderProducer("unused", "fake-topic", producer_client=fake_pressure)
service.publish(make_order_event(2003))
summary = service.finish()
assert fake_pressure.queue_attempts == 3
assert summary.queued == 1
assert summary.all_confirmed
print("PASS: local queue pressure retried; no duplicate accepted send")

### Edge case: `flush()` times out without a final callback

**Objective:** Demonstrate why **outstanding** is a third state, different from both delivered and failed. If shutdown times out, those messages are still unresolved from the application's perspective. Do not count them as success.

In [ ]:
class FakeStillPending(FakeProducer):
    """Simulate a flush timeout with a message still outstanding."""

    def flush(self, timeout=35):
        return len(self.pending)  # No callback processed yet.


never_finished = FakeStillPending()
service = CallbackConfirmedOrderProducer(
    "unused", "fake-topic", producer_client=never_finished
)
service.publish(make_order_event(2004))
summary = service.finish(timeout=0.01)

assert summary.delivered == 0
assert summary.failed == 0
assert summary.awaiting_callback == 1
assert summary.producer_remaining == 1
assert not summary.all_confirmed
print("PASS: unresolved work is not mislabeled as success")

### What we proved (and what we didn't)

These unit tests prove that **our application logic**:

- Doesn't confuse local acceptance with confirmed delivery.
- Separates successful and failed final delivery outcomes.
- Handles temporary `BufferError` without counting rejected attempts as queued.

They do **not** prove broker replication, the real network retry protocol, message ordering, or downstream consumer behavior. Those require integration and system tests.

---
# Section 11 — Common mistakes and production decisions

**Objective:** Recognize unsafe shortcuts before they reach production.

| Common mistake | Why it's wrong | Better approach |
|---|---|---|
| Call `flush()` after every `produce()` | Makes an async pattern behave synchronously; hurts throughput | Poll during publishing, flush at shutdown |
| Treat `produce()` returning as proof of delivery | It only confirms local acceptance | Track final delivery callback |
| Treat `flush()==0` as 100% success | Failed deliveries also finish | Check delivery failures and unresolved counts |
| Never service producer events | Callbacks/diagnostics can build up or go unprocessed | Call `poll()` periodically or use a dedicated serviced loop |
| Retry `produce()` forever on `BufferError` | Can hang an API or worker indefinitely | Apply backpressure with a deadline |
| Blindly resend every failed callback | Ambiguous outcomes can create duplicate business events | Stable IDs, downstream dedupe, durable recovery workflow |
| Heavy database/network work inside callback | Blocks event servicing and increases latency | Quick metrics/logging; defer heavy work |
| Create a producer for every event | Wastes connections, batching, memory | Reuse a long-lived producer |
| Think `acks=all` means all consumers processed the event | Acknowledgement is about broker receipt/durability policy | Track consumer/business outcome separately |

## Decision guide: Is callback-confirmed publishing enough?

- **Log/telemetry:** Fire-and-forget may be sufficient if occasional loss is acceptable.
- **Standard service event:** Delivery callbacks + bounded retry + metrics may meet requirements.
- **Order/payment/database state-change event:** Callback confirmation is valuable, but cannot make the preceding database commit and Kafka publish atomic. Consider a **transactional outbox**, downstream idempotency, and end-to-end monitoring.

**Kafka has important reliability features; application-level exactly-once business effects are a separate design problem.**

---
# Section 12 — Exercises: from junior to independent engineer

**Objective:** Use the concepts without copying the earlier cells.

### Exercise A — Beginner (5 minutes)

Add `on_delivery` to a producer publishing three order events. Print either `FAILED` or the successful `partition` and `offset`.

**Pass condition:** You can explain why printing immediately after `produce()` is not a broker acknowledgement.

### Exercise B — Intermediate (10 minutes)

Publish 100 events with a callback and `poll(0)` inside the loop; call `flush(20)` once after the loop. Track `queued`, `delivered`, `failed`, and `awaiting_callback`.

**Pass condition:** Your counters reconcile when all delivery callbacks complete, regardless of successes or failures.

### Exercise C — Failure handling (10 minutes)

Change a *separate test producer* to an unreachable bootstrap address with a 4-second delivery timeout. Observe a failed callback.

**Pass condition:** You can distinguish a synchronous `BufferError` from an asynchronous failed-delivery callback.

### Exercise D — Production judgment (discussion)

Your order service commits an order row to PostgreSQL, then publishes `order.created`. It crashes after the DB commit but before the Kafka send. Would a delivery callback solve this gap?

**Answer:** No. A callback only covers messages that the producer submitted and tracked. Investigate a transactional outbox so the DB change and an outbox entry are stored in the same DB transaction, then publish the outbox entry reliably.

---
# Section 13 — The one-page engineer's cheat sheet

**If you forget the entire notebook, remember only this:**

```text
produce(event, on_delivery=callback)   -> QUEUE (accepted locally)
background networking                 -> SEND (happens automatically)
Kafka broker                           -> ACK (according to acks policy)
poll(0)                                -> REPORT (run ready callbacks)
flush(timeout)                         -> DRAIN (wait before shutdown)
```

### The standard callback-confirmed loop

```python
producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
})


def on_delivery(error, message):
    if error:
        print("FAILED:", error)  # In production: structured log/metric/recovery.
    else:
        print("ACK:", message.partition(), message.offset())


try:
    for event in events:
        producer.produce(
            topic="orders",
            key=str(event["order_id"]),
            value=json.dumps(event).encode("utf-8"),
            on_delivery=on_delivery,
        )
        producer.poll(0)
finally:
    remaining = producer.flush(timeout=30)
    if remaining:
        print("Unresolved producer work:", remaining)
```

*Above is the conceptual loop; real services also handle `BufferError`, observe callback failures, and apply recovery/shutdown policy. Our `CallbackConfirmedOrderProducer` in Section 8 implements those missing basics.*

### Six interview-ready answers

1. **What is callback-confirmed publishing?** Asynchronous publishing that observes final per-message delivery outcomes via callbacks.
2. **Does `produce()` send over the network directly?** It enqueues locally; background networking performs transmission.
3. **What runs the callback?** Calls to `producer.poll()` or `producer.flush()` normally serve it.
4. **What's the difference between queue-full and failed delivery?** Queue-full rejects local enqueue; failed delivery is the terminal result for a previously accepted message.
5. **Does `flush()==0` mean all messages succeeded?** No; inspect failures too.
6. **Does idempotence guarantee exactly-once business processing?** No; it protects against certain producer retry duplicates, not cross-system atomicity or consumer side effects.

### When you're ready for the next pattern

Explore batching, idempotent producer internals, transactions, or the **transactional outbox**. Do not introduce those abstractions until the basic callback lifecycle feels natural.

---
## Official references

- [Confluent Python Client API — Producer `produce()`, `poll()`, `flush()`](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
- [Confluent's official Python producer example](https://github.com/confluentinc/confluent-kafka-python/blob/master/examples/producer.py)
- [Confluent Python Client README and callback examples](https://github.com/confluentinc/confluent-kafka-python)
- [Kafka Producer Configuration Reference](https://docs.confluent.io/platform/current/installation/configuration/producer-configs.html)

**Notebook version:** October 2026. Verify configuration defaults against your deployed Kafka/client versions, especially on managed clusters.